#  piNEUMA Parser for Local Folder
This notebook loads and parses **all piNEUMA CSV files** from the local `data/` folder.

In [1]:
# 1) Import libraries
import pandas as pd
import numpy as np
import os
import glob

In [2]:
# 2) Set path to your local folder
folder_path = '../data'  # path relative to notebooks folder
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
dataframes = {}

Parse each file into a labeled dataframe

In [3]:

# 3) Parse each file into a labeled DataFrame
for file_path in csv_files:
    filename = os.path.basename(file_path)
    parts = filename.split('_')

    if len(parts) < 3:
        print(f"Skipping file with unexpected format: {filename}")
        continue

    date_str = parts[0]
    location_str = parts[1]
    start_time_str = parts[2]
    monthday = date_str[4:]

    loc_num = location_str.replace("d", "")
    loc_label = f"L{loc_num}"

    if start_time_str.startswith("0"):
        start_time_str = start_time_str[1:]

    var_name = f"df_{loc_label}{start_time_str}_{monthday}"

    rows = []
    with open(file_path, 'r', encoding='utf-8') as f:
        next(f)
        for line_num, line in enumerate(f, start=2):
            if line_num > 2000:
                break
            fields = line.strip().split(';')
            if len(fields) < 4:
                continue

            track_id    = fields[0]
            vehicle_type = fields[1]
            traveled_d  = fields[2]
            avg_speed   = fields[3]
            time_step_fields = fields[4:]

            while time_step_fields and not time_step_fields[-1].strip():
                time_step_fields.pop()

            leftover = len(time_step_fields) % 6
            if leftover != 0:
                time_step_fields = time_step_fields[:-leftover]

            n_blocks = len(time_step_fields) // 6

            for i in range(n_blocks):
                offset = i * 6
                lat     = time_step_fields[offset + 0]
                lon     = time_step_fields[offset + 1]
                speed   = time_step_fields[offset + 2]
                lon_acc = time_step_fields[offset + 3]
                lat_acc = time_step_fields[offset + 4]
                tstamp  = time_step_fields[offset + 5]
                rows.append({
                    "track_id":   track_id,
                    "type":       vehicle_type,
                    "traveled_d": traveled_d,
                    "avg_speed":  avg_speed,
                    "lat":        lat,
                    "lon":        lon,
                    "speed":      speed,
                    "lon_acc":    lon_acc,
                    "lat_acc":    lat_acc,
                    "time":       tstamp,
                    "time_step":  i
                })

    df = pd.DataFrame(rows)
    numeric_cols = ["traveled_d", "avg_speed", "lat", "lon", "speed", "lon_acc", "lat_acc", "time"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    dataframes[var_name] = df
    print(f"Parsed {filename} -> {var_name} with shape: {df.shape}")


Parsed 20181024_dX_0830_0900 (1).csv -> df_LX830_1024 with shape: (8333923, 11)
Parsed 20181024_dX_1030_1100.csv -> df_LX1030_1024 with shape: (8743536, 11)


In [4]:
# One example
list(dataframes.keys())

['df_LX830_1024', 'df_LX1030_1024']

Preview dataframes

In [5]:
# 4)Preview one of the parsed DataFrames
sample_key = list(dataframes.keys())[0]
dataframes[sample_key].head()

,track_id,type,traveled_d,avg_speed,lat,lon,speed,lon_acc,lat_acc,time,time_step
0,1,Car,10.18,36.634649,37.984642,23.724906,38.1611,0.0,0.0,0.00,0
1,1,Car,10.18,36.634649,37.984643,23.724901,38.1611,0.0,0.0,0.04,1
2,1,Car,10.18,36.634649,37.984643,23.724896,38.1611,0.0,0.0,0.08,2
3,1,Car,10.18,36.634649,37.984644,23.724892,38.1611,0.0,0.0,0.12,3
4,1,Car,10.18,36.634649,37.984645,23.724887,38.1611,0.0,0.0,0.16,4


Check vehicle type count and percentage 

In [ ]:
# check vehicle_type % and count
for name, df in dataframes.items():
    print(f"\n Analyzing: {name}")
    
    unique_vehicles = df.drop_duplicates(subset=["track_id"])
    vehicle_type_counts = unique_vehicles["type"].value_counts()
    vehicle_type_percentage = unique_vehicles["type"].value_counts(normalize=True) * 100

    print("Vehicle Type Counts:")
    print(vehicle_type_counts)

    print("\nVehicle Type Percentages (%):")
    print(vehicle_type_percentage)

In [ ]:
Flow

In [ ]:
bin_width = 10  # seconds

for name, df in dataframes.items():
    print(f"\n Flow analysis for: {name}")
    
    # Create a time bin column
    df["time_bin"] = (df["time"] // bin_width).astype(int)
    
    # Count unique vehicles in each time bin
    flow_df = df.groupby("time_bin")["track_id"].nunique().reset_index(name="unique_vehicle_count")
    
    # Convert to flow in vehicles/hour
    flow_df["flow"] = flow_df["unique_vehicle_count"] * (3600 / bin_width)
    
    print(flow_df.head())

Density

In [ ]:
bin_width = 10  # seconds
segment_length_km = 1.0  # assumed constant for now

for name, df in dataframes.items():
    print(f"\n📊 Flow + Density analysis for: {name}")
    
    df["time_bin"] = (df["time"] // bin_width).astype(int)
    
    flow_df = df.groupby("time_bin")["track_id"].nunique().reset_index(name="unique_vehicle_count")
    flow_df["flow"] = flow_df["unique_vehicle_count"] * (3600 / bin_width)
    flow_df["density"] = flow_df["unique_vehicle_count"] / segment_length_km
    
    print(flow_df[["time_bin", "flow", "density"]].head())
    

Average speed

In [ ]:
bin_width = 10  # seconds

for name, df in dataframes.items():
    print(f"\n🚦 Speed analysis for: {name}")
    
    # Time bin column (in case it's not there yet)
    df["time_bin"] = (df["time"] // bin_width).astype(int)

    # Average speed per vehicle
    vehicle_avg = df.groupby("track_id")["speed"].mean().reset_index(name="vehicle_avg_speed")
    unique_vehicle_info = df.drop_duplicates(subset=["track_id"])[["track_id", "type"]]
    vehicle_avg = vehicle_avg.merge(unique_vehicle_info, on="track_id", how="left")

    # Average speed by vehicle type
    avg_speed_by_type = vehicle_avg.groupby("type")["vehicle_avg_speed"].mean().reset_index()
    print("📊 Average Speed by Vehicle Type:")
    print(avg_speed_by_type)

    # Average speed per time bin
    speed_df = df.groupby("time_bin")["speed"].mean().reset_index(name="avg_speed")
    print("\n📈 Average Speed per Time Bin (km/h):")
    print(speed_df.head())

Time statistics

In [ ]:
for name, df in dataframes.items():
    print(f"\n⏱ Time stats for: {name}")
    
    df = df.copy()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")

    print(df[["track_id", "time"]].head())  # optional preview

    print("Min time:", df["time"].min())
    print("Max time:", df["time"].max())
    print("Total duration (s):", df["time"].max() - df["time"].min())

    print("\n" + "-"*40)

Step-by-step implementation

In [ ]:
from geopy.distance import distance
import matplotlib.pyplot as plt

def in_boxspace(lat, lon, lat_min, lat_max, lon_min, lon_max):
    return (lat_min <= lat <= lat_max) and (lon_min <= lon <= lon_max)

def crosses_line(lat1, lat2, lat_line):
    return (lat1 < lat_line < lat2) or (lat2 < lat_line < lat1)

def compute_fundamental_data(df, lat_min, lat_max, lon_min, lon_max,
                             lat_upstream, corridor_length_km,
                             t_start, t_end, bin_seconds=30):

    df = df.sort_values(["time", "track_id"]).reset_index(drop=True)
    results = []
    current_t = t_start
    df_time_filtered = df[(df["time"] >= t_start) & (df["time"] < t_end)]

    while current_t < t_end:
        bin_start = current_t
        bin_end   = current_t + bin_seconds

        bin_df = df_time_filtered[
            (df_time_filtered["time"] >= bin_start) &
            (df_time_filtered["time"] < bin_end)
        ]

        # Density
        in_box_mask = (
            (bin_df["lat"] >= lat_min) & (bin_df["lat"] <= lat_max) &
            (bin_df["lon"] >= lon_min) & (bin_df["lon"] <= lon_max)
        )
        in_box_df = bin_df[in_box_mask]
        density_veh = in_box_df["track_id"].nunique()
        density = density_veh / corridor_length_km

        # Flow
        crossing_vehicles = set()
        grouped_veh = bin_df.groupby("track_id", sort=False)
        for veh_id, group in grouped_veh:
            lat_values = group["lat"].values
            for i in range(len(lat_values) - 1):
                if crosses_line(lat_values[i], lat_values[i+1], lat_upstream):
                    crossing_vehicles.add(veh_id)
                    break

        flow_count = len(crossing_vehicles)
        flow = flow_count / (bin_seconds / 3600.0)

        avg_speed = in_box_df["speed"].mean() if len(in_box_df) > 0 else 0.0
        results.append([bin_start, density, flow, avg_speed])
        current_t += bin_seconds

    return pd.DataFrame(results, columns=["time_bin", "density", "flow", "speed"])

Apply all dataframes

In [ ]:
for name, df in dataframes.items():
    print(f"\n📊 Fundamental Diagram for: {name}")
    df = df.copy()
    df["time"] = pd.to_numeric(df["time"], errors="coerce")

    # Bounding box
    lat_min = df["lat"].min()
    lat_max = df["lat"].max()
    lon_min = df["lon"].min()
    lon_max = df["lon"].max()

    # Distance-based segment length
    corner1 = (lat_min, lon_min)
    corner2 = (lat_max, lon_max)
    approx_meters = distance(corner1, corner2).meters
    corridor_length_km = approx_meters / 1000.0
    lat_upstream = (lat_min + lat_max) / 2.0

    # Time range
    t_min = df["time"].min()
    t_max = df["time"].max()
    time_start = t_min
    time_end = min(t_min + 1800, t_max)  # 30 minutes max

    # Compute FD data
    fd_data = compute_fundamental_data(
        df,
        lat_min, lat_max, lon_min, lon_max,
        lat_upstream,
        corridor_length_km,
        t_start=time_start,
        t_end=time_end,
        bin_seconds=30
    )

    print(fd_data.head())

    # Plot 1: Flow vs Density
    plt.figure(figsize=(7, 5))
    plt.scatter(fd_data["density"], fd_data["flow"], alpha=0.7)
    plt.title(f"Flow vs Density – {name}")
    plt.xlabel("Density (vehicles/km)")
    plt.ylabel("Flow (vehicles/hour)")
    plt.grid(True)
    plt.show()

    # Plot 2: Flow vs Speed
    plt.figure(figsize=(7, 5))
    plt.scatter(fd_data["flow"], fd_data["speed"], alpha=0.7)
    plt.title(f"Speed vs Flow – {name}")
    plt.xlabel("Flow (vehicles/hour)")
    plt.ylabel("Speed (km/h)")
    plt.grid(True)
    plt.show()